# 02 · Payment Abuse Model (multi-class LightGBM)

**FraudShield AI** · Model 2. Detects trial abuse, discount / coupon abuse, shared-card abuse, promo farming, and payment fraud.

- Classes: `0 = legit`, `1 = trial_abuse`, `2 = discount_abuse`, `3 = payment_fraud`.
- Consumes the upstream `trust_score` plus payment, card, device, IP, velocity, and graph features.
- Output: `payment_risk_score` (0-100), `abuse_type`, and a `decision`.

In [ ]:
from trust_radar.config import PAYMENT_LABELS, FeatureConfig, PaymentModelConfig
from trust_radar.inference.predict_payment import PaymentPredictor
from trust_radar.training.train_payment import train_payment_model
from trust_radar.utils.synthetic import synthesize_payment_dataset

## 1. Synthesize a multi-class payment dataset

In [ ]:
cfg = FeatureConfig()
df = synthesize_payment_dataset(n=6000, seed=42)

print('classes:', PAYMENT_LABELS)
print(df['label'].map(PAYMENT_LABELS).value_counts())
print()
print('feature groups:', {k: len(v) for k, v in cfg.payment_feature_groups.items()})
df.head()

## 2. Prepare features

Categorical columns use the pandas `category` dtype so LightGBM handles them natively; identifier columns are excluded.

In [ ]:
X = df[cfg.payment_features].copy()
for col in cfg.payment_categorical_features:
    X[col] = X[col].astype('category')
y = df['label']
X.shape, y.nunique()

## 2b. Train / test split (stratified)

`train_payment_model()` (next section) performs this same stratified 80/20 split internally. Doing it explicitly here first lets you inspect split sizes and confirm every abuse class is represented in both train and test before any model code runs -- this is the *test data* the model will be scored on.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('train rows:', len(X_train))
print('test  rows:', len(X_test))
print()
print('train class counts:')
print(y_train.map(PAYMENT_LABELS).value_counts())
print()
print('test class counts:')
print(y_test.map(PAYMENT_LABELS).value_counts())

In [ ]:
import matplotlib.pyplot as plt

train_counts = y_train.map(PAYMENT_LABELS).value_counts().reindex(PAYMENT_LABELS.values())
test_counts = y_test.map(PAYMENT_LABELS).value_counts().reindex(PAYMENT_LABELS.values())

fig, ax = plt.subplots(figsize=(8, 4.5))
x_pos = range(len(PAYMENT_LABELS))
width = 0.35
ax.bar([p - width / 2 for p in x_pos], train_counts.values, width, label='train', color='#4C72B0')
ax.bar([p + width / 2 for p in x_pos], test_counts.values, width, label='test', color='#DD8452')
ax.set_xticks(list(x_pos))
ax.set_xticklabels(list(PAYMENT_LABELS.values()), rotation=20, ha='right')
ax.set_ylabel('rows')
ax.set_title('Class balance: train vs. test')
ax.legend()
fig.tight_layout()
plt.show()

### Early-stopping search for `n_estimators`

The multi-class model does **not** train in "epochs" -- LightGBM adds boosting rounds (trees), controlled by `PaymentModelConfig.n_estimators` (currently 300 by default). Rather than guessing that number, this cell fits an *uncalibrated* LightGBM directly on a train/validation split with early stopping, so it picks the number of rounds where validation log-loss stops improving. That number is then reused for the real, calibrated training run below.

(Note: `PaymentAbuseModel.fit(calibrate=True)` wraps LightGBM in `CalibratedClassifierCV`, which does its own internal 3-fold CV and does not support early stopping -- that is why this search uses the raw estimator with `calibrate=False` first.)

In [ ]:
import lightgbm as lgb

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

search_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=48,
    class_weight='balanced',
    random_state=42,
    verbosity=-1,
)
search_model.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    eval_metric='multi_logloss',
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
)

best_n_estimators = search_model.best_iteration_
print('best_iteration_ (n_estimators to use):', best_n_estimators)

In [ ]:
eval_curve = search_model.evals_result_['valid_0']['multi_logloss']

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(eval_curve, color='#4C72B0')
ax.axvline(best_n_estimators, color='#C44E52', linestyle='--', label=f'best_iteration={best_n_estimators}')
ax.set_xlabel('boosting round')
ax.set_ylabel('validation multi_logloss')
ax.set_title('Early-stopping curve')
ax.legend()
fig.tight_layout()
plt.show()

## 3. Train and calibrate the multi-class model

In [ ]:
model, metrics = train_payment_model(
    X, y,
    config=PaymentModelConfig(n_estimators=best_n_estimators),
    categorical_features=cfg.payment_categorical_features,
)
print('accuracy          :', round(metrics['accuracy'], 4))
print('macro_f1          :', round(metrics['macro_f1'], 4))
print('macro_roc_auc_ovr :', round(metrics['macro_roc_auc_ovr'], 4))
print('abuse_roc_auc     :', round(metrics['abuse_roc_auc'], 4))

### Per-class quality

In [ ]:
for name in PAYMENT_LABELS.values():
    print(f"{name:16s} "
          f"P={metrics['precision_' + name]:.3f} "
          f"R={metrics['recall_' + name]:.3f} "
          f"F1={metrics['f1_' + name]:.3f}")

### Test-set confusion matrix & per-class ROC

`train_payment_model()` evaluates on its own internal 80/20 test split (same stratified logic as `X_test`/`y_test` above). These graphs recompute predictions on that same held-out test data to visualize where the model confuses abuse types.

In [ ]:
from trust_radar.utils.metrics import multiclass_confusion

test_risk = model.predict_risk(X_test)
cm = multiclass_confusion(y_test.to_numpy(), test_risk['predicted_class'], n_classes=4)

fig, ax = plt.subplots(figsize=(6, 5.5))
im = ax.imshow(cm, cmap='Blues')
labels = list(PAYMENT_LABELS.values())
ax.set_xticks(range(4)); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_yticks(range(4)); ax.set_yticklabels(labels)
ax.set_xlabel('predicted'); ax.set_ylabel('actual')
ax.set_title('Test-set confusion matrix')
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i][j], ha='center', va='center',
                color='white' if cm[i][j] > (max(max(r) for r in cm) / 2) else 'black')
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import auc, roc_curve

fig, ax = plt.subplots(figsize=(7, 5.5))
y_test_np = y_test.to_numpy()
for class_idx, name in PAYMENT_LABELS.items():
    y_true_bin = (y_test_np == class_idx).astype(int)
    fpr, tpr, _ = roc_curve(y_true_bin, test_risk['probabilities'][:, class_idx])
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
ax.set_xlabel('false positive rate'); ax.set_ylabel('true positive rate')
ax.set_title('Test-set per-class ROC (one-vs-rest)')
ax.legend()
fig.tight_layout()
plt.show()

## 4. Feature importance

The upstream `trust_score` and the card-reputation / velocity features should dominate.

In [ ]:
model.get_feature_importances().head(15)

## 5. Real-time decision scoring with plan gating

Full-price plans skip the model and are always allowed; trial and discounted plans are scored.

In [ ]:
predictor = PaymentPredictor(model_or_path=model)

full_price = predictor.score_transaction(df.iloc[[0]], plan_type='full_price')
print('full price (gated):', full_price)

trial = predictor.score_transaction(df.iloc[[1]], plan_type='trial')
print('trial (scored)   :', trial)

## 6. Batch scoring & decision distribution

In [ ]:
scored = predictor.score_batch(df.head(500))
print('Decisions:')
print(scored['decision'].value_counts())
print()
print('Predicted abuse types:')
print(scored['abuse_type'].value_counts())
scored[['plan_type', 'payment_risk_score', 'risk_level', 'abuse_type', 'decision']].head(10)

## 7. Decision logic

| Payment risk score | Action |
|--------------------|--------|
| 0-40 | `ALLOW` |
| 41-70 | `ALLOW_FLAG_REVIEW` |
| 71-94 | `ALLOW_HIGH_PRIORITY_REVIEW` |
| 95-100 | `BLOCK` (block trial / block discount) |
